# Fetch API 
- No API-key requiered

In [7]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
CONTACT = os.getenv("MB_CONTACT")

url = "https://musicbrainz.org/ws/2/recording"
headers = {"User-Agent": f"MusikQuizApp/0.1 ({CONTACT})"}
params = {
    "query": "tag:rock AND date:[1980 TO 1989]",
    "limit": 25,
    "fmt": "json",
}

r = requests.get(url, params=params, headers=headers, timeout=30)
r.raise_for_status()

df = pd.json_normalize(r.json()["recordings"])
df.columns

Index(['id', 'score', 'artist-credit-id', 'title', 'length', 'disambiguation',
       'video', 'artist-credit', 'first-release-date', 'releases', 'isrcs',
       'tags'],
      dtype='str')

### Fetch data using function

In [3]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads in .env

BASE_URL = "https://musicbrainz.org/ws/2"
CONTACT = os.getenv("MB_CONTACT")
if not CONTACT:
    raise RuntimeError("MB_CONTACT missing in .env")

HEADERS = {"User-Agent": f"MusikQuizApp/0.1 ({CONTACT})"}

def mb_get(endpoint, params):
    params = {**params, "fmt": "json"}
    r = requests.get(f"{BASE_URL}/{endpoint}", params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    time.sleep(1)  # rate limit
    return r.json()

# Ex: recordnings from the 80's by a certain artist
data = mb_get("recording", {
    "query": 'artist:"a-ha" AND date:[1980 TO 1989]',
    "limit": 100,
    "offset": 0,
})

print(data["count"])
print(data["recordings"][0])

100
{'id': '09a744e3-04b2-4013-92c0-477b7a46e1c8', 'score': 100, 'artist-credit-id': '07885c4d-864e-30be-ba2b-541c03409c3b', 'title': 'Manhattan Skyline', 'length': 284000, 'video': None, 'artist-credit': [{'name': 'a‐ha', 'artist': {'id': '7364dea6-ca9a-48e3-be01-b44ad0d19897', 'name': 'a‐ha', 'sort-name': 'a‐ha', 'disambiguation': 'Norwegian synth‐pop band', 'aliases': [{'sort-name': 'A-ha', 'name': 'A-ha', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'A Ha', 'name': 'A Ha', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'Aha', 'name': 'Aha', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}]}}], 'first-release-date': '1987-03-23', 'releases': [{'id': 'd958caa4-3944-4085-a0ee-7c3aaaed6597', 'status-id': '4e304316-386d-3409-af2e-78857eec5cfe', 'artist-credit-id': '949a7fd5-fe73-3e8f-922e-01ff4ca958f7', 'count': 2, 'title': 'Now That’s What I Ca

### Some EDA artist "a-ha" between dates 1980 - 1989

In [ ]:
# Check columns 
df = pd.json_normalize(data["recordings"])
df.columns

Index(['id', 'score', 'artist-credit-id', 'title', 'length', 'video',
       'artist-credit', 'first-release-date', 'releases', 'tags',
       'disambiguation', 'isrcs'],
      dtype='str')

In [12]:
# first rows
df.head()

,id,score,artist-credit-id,title,length,video,artist-credit,first-release-date,releases,tags,disambiguation,isrcs
0,09a744e3-04b2-4013-92c0-477b7a46e1c8,100,07885c4d-864e-30be-ba2b-541c03409c3b,Manhattan Skyline,284000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1987-03-23,[{'id': 'd958caa4-3944-4085-a0ee-7c3aaaed6597'...,"[{'count': 1, 'name': 'soundtrack'}, {'count':...",NaN,NaN
1,4cae7d8d-18d5-4591-90c6-d7d0b7e26026,100,07885c4d-864e-30be-ba2b-541c03409c3b,October,228000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1986,[{'id': 'cb93cdcf-f173-49d1-b18a-e2264fcb3058'...,NaN,NaN,NaN
2,1c3aba04-c20a-49ab-9943-bfd69c7e32f7,100,07885c4d-864e-30be-ba2b-541c03409c3b,Soft Rains of April = Lluvias Suaves de Abril,192000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1986,[{'id': 'e538f28e-097b-4ce4-829a-9213744a1ef7'...,NaN,NaN,NaN
3,a5c4ce4f-2bb2-4a1b-8e6e-5f94163698a7,100,07885c4d-864e-30be-ba2b-541c03409c3b,Take on Me,NaN,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1986,[{'id': '4e045f76-dd88-494a-8be4-6b7078dc0366'...,NaN,NaN,NaN
4,4724b050-d5da-416f-9f09-2c18977e4f8d,100,07885c4d-864e-30be-ba2b-541c03409c3b,Cry Wolf,245000.0,None,"[{'name': 'a‐ha', 'artist': {'id': '7364dea6-c...",1986,[{'id': 'cb93cdcf-f173-49d1-b18a-e2264fcb3058'...,NaN,NaN,NaN


In [11]:
# types and number of missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  100 non-null    str    
 1   score               100 non-null    int64  
 2   artist-credit-id    100 non-null    str    
 3   title               100 non-null    str    
 4   length              95 non-null     float64
 5   video               0 non-null      object 
 6   artist-credit       100 non-null    object 
 7   first-release-date  100 non-null    str    
 8   releases            100 non-null    object 
 9   tags                7 non-null      object 
 10  disambiguation      11 non-null     str    
 11  isrcs               2 non-null      object 
dtypes: float64(1), int64(1), object(5), str(5)
memory usage: 19.6+ KB


In [13]:
# number of rows and columns
df.shape

(100, 12)

### Check with another artist and years

In [16]:
data = mb_get("recording", {
    "query": 'artist:"Britney Spears" AND date:[1995 TO 2005]',
    "limit": 100,
    "offset": 0,
})

print(data["count"])
print(data["recordings"][0])

488
{'id': 'b89f48bb-5dce-4ce3-aec9-364477f526fa', 'score': 100, 'artist-credit-id': '038401cc-de8b-3013-9efe-4ee3e07782f6', 'title': 'Boys', 'length': 242000, 'disambiguation': 'part of “The Very Best of Pure R&B: The Summer Collection” DJ‐mix', 'video': None, 'artist-credit': [{'name': 'Britney Spears', 'artist': {'id': '45a663b5-b1cb-4a91-bff6-2bef7bbfdd76', 'name': 'Britney Spears', 'sort-name': 'Spears, Britney', 'aliases': [{'sort-name': '브리트니 스피어스', 'type-id': '1937e404-b981-3cb7-8151-4c86ebfc8d8e', 'name': '브리트니 스피어스', 'locale': None, 'type': 'Search hint', 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'ブリトニー・スピアーズ', 'type-id': '894afba6-2816-3c24-8072-eadb66bd04bc', 'name': 'ブリトニー・スピアーズ', 'locale': 'ja', 'type': 'Artist name', 'primary': True, 'begin-date': None, 'end-date': None}, {'sort-name': 'Bretney Spears', 'name': 'Bretney Spears', 'locale': None, 'type': None, 'primary': None, 'begin-date': None, 'end-date': None}, {'sort-name': 'B. Spears', 'na

In [17]:
df = pd.json_normalize(data["recordings"])
df.columns

Index(['id', 'score', 'artist-credit-id', 'title', 'length', 'disambiguation',
       'video', 'artist-credit', 'first-release-date', 'releases', 'isrcs',
       'tags'],
      dtype='str')

In [18]:
df.head()

,id,score,artist-credit-id,title,length,disambiguation,video,artist-credit,first-release-date,releases,isrcs,tags
0,b89f48bb-5dce-4ce3-aec9-364477f526fa,100,038401cc-de8b-3013-9efe-4ee3e07782f6,Boys,242000.0,part of “The Very Best of Pure R&B: The Summer...,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",2003-03-31,[{'id': 'dfeca7a0-f9c8-48e3-b609-24debb4769be'...,NaN,NaN
1,5286dc3a-c699-4784-8bde-12701932f0fd,100,038401cc-de8b-3013-9efe-4ee3e07782f6,(You Drive Me) Crazy (The Stop Remix!) (instru...,199400.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",1999-09-29,[{'id': '9eb6afce-a06c-468c-8bca-1408a8881f0f'...,[USJI19910720],"[{'count': 1, 'name': 'electronic'}, {'count':..."
2,fb975c61-c934-4880-835e-d23437401b25,100,038401cc-de8b-3013-9efe-4ee3e07782f6,(You Drive Me) Crazy (Jazzy Jim’s hip-hop mix),220000.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",1999-12-06,[{'id': 'acaee915-0ecd-4e3a-a112-be1d2e6d9641'...,NaN,"[{'count': 3, 'name': 'pop'}, {'count': 1, 'na..."
3,d9a51139-373d-48ab-9539-2af59860a2b3,100,038401cc-de8b-3013-9efe-4ee3e07782f6,Right Now (Taste the Victory),314760.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",2002-04,[{'id': '54593f6e-5251-439c-a45b-102b940773da'...,NaN,NaN
4,2c4e741b-b0d0-452a-994e-226925516632,100,038401cc-de8b-3013-9efe-4ee3e07782f6,Toxic,186000.0,NaN,None,"[{'name': 'Britney Spears', 'artist': {'id': '...",2004,[{'id': '3ef19145-4c5a-4ede-8d0f-1c9a4c3c225b'...,NaN,NaN
